In [120]:
!pip install transformers datasets peft accelerate bitsandbytes


   ---------------------------------------- 0.0/557.0 kB ? eta -:--:--
   ---------------------------------------- 557.0/557.0 kB 9.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/55.4 MB ? eta -:--:--
   --- ------------------------------------ 4.5/55.4 MB 20.7 MB/s eta 0:00:03
   ------ --------------------------------- 8.9/55.4 MB 20.5 MB/s eta 0:00:03
   ---------- ----------------------------- 15.2/55.4 MB 23.9 MB/s eta 0:00:02
   ---------------- ----------------------- 22.8/55.4 MB 26.2 MB/s eta 0:00:02
   ---------------------- ----------------- 31.2/55.4 MB 28.7 MB/s eta 0:00:01
   ---------------------------- ----------- 38.8/55.4 MB 30.4 MB/s eta 0:00:01
   ---------------------------------- ----- 47.4/55.4 MB 31.8 MB/s eta 0:00:01
   ---------------------------------------  55.3/55.4 MB 33.2 MB/s eta 0:00:01
   ---------------------------------------- 55.4/55.4 MB 30.9 MB/s eta 0:00:00


In [135]:
from datasets import load_dataset

dataset = load_dataset("zillow/real_estate_v1")["train"]

print(len(dataset))
print(dataset.column_names)

21382
['messages', 'split', 'topic']


In [136]:
def flatten(example):
    return {
        "text": " ".join([m["content"] for m in example["messages"]])
    }

dataset = dataset.map(flatten)
dataset = dataset.remove_columns(["messages", "split", "topic"])

print(dataset.column_names)
print(len(dataset))

['text']
21382


In [137]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/21382 [00:00<?, ? examples/s]

In [139]:
tokenized = tokenized.remove_columns(["text"])
tokenized.set_format("torch")

print(tokenized.column_names)
print(len(tokenized))

['input_ids', 'attention_mask', 'labels']
21382


In [140]:
split_dataset = tokenized.train_test_split(test_size=0.1)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(len(train_dataset), len(eval_dataset))

19243 2139


In [144]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("distilgpt2")

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    logging_steps=100,
    save_strategy="no",
    remove_unused_columns=False
)

In [143]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

In [145]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.486900,2.360421
2,2.424500,2.318690


TrainOutput(global_step=4812, training_loss=2.558899721698967, metrics={'train_runtime': 653.5004, 'train_samples_per_second': 58.892, 'train_steps_per_second': 7.363, 'total_flos': 1261391736471552.0, 'train_loss': 2.558899721698967, 'epoch': 2.0})

In [146]:
import math

eval_results = trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])

print("Eval Loss:", eval_results["eval_loss"])
print("Perplexity:", perplexity)

Eval Loss: 2.318690299987793
Perplexity: 10.1623559487406


In [149]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained("distilgpt2")

base_trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

base_eval = base_trainer.evaluate()

import math
base_ppl = math.exp(base_eval["eval_loss"])

print("Base Perplexity:", base_ppl)

Base Perplexity: 26.624256038600272


In [152]:
prompt = """
Below is a real estate conversation.

User: How do agents handle low home appraisals?
Assistant:
"""

In [156]:
inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.4
)

print("Fine-tuned Output:\n")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Fine-tuned Output:


Below is a real estate conversation.

User: How do agents handle low home appraisals?
Assistant:
User: How do agents handle low home appraisals?
Assistant: How do agents handle low home appraisals?
Assistant: How do agents handle low home appraisals?
Assistant: How do agents handle low home appraisals?
Assistant: How do agents handle low home appraisals?
Assistant: How do agents handle low home appraisals?
Assistant: How do agents handle low home appraisals?
Assistant: How do agents handle low home appra


In [151]:
inputs = tokenizer(prompt, return_tensors="pt")

output = base_model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.7
)

print("Base Model Output:\n")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
c:\Users\viole\anaconda3\Lib\site-packages\transformers\generation\utils.py:2208: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)